# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**1. Ranked Actions and Reason Codes**

*   **Priority 1: Content Refresh (Reason Code: R-01)**
    *   **Trigger:** High Impressions (>500), Avg Position <= 15, CTR < 0.01
    *   **Action:** Send to content team for metadata optimization (Title/Meta Description rewrite) to improve clickability.
*   **Priority 2: Technical SEO Audit (Reason Code: R-02)**
    *   **Trigger:** High Impressions, Avg Position 1-3, CTR < 0.05
    *   **Action:** Flag for technical review. High rank but low CTR usually indicates SERP layout issues (e.g., zero-click searches, heavy ads) or brand mismatch.
*   **Priority 3: Prune / Consolidate (Reason Code: R-03)**
    *   **Trigger:** Content Age > 365 days, Zero Clicks in last 3 months, Low Impressions.
    *   **Action:** Evaluate for redirection (301) to a fresher, broader pillar page.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**2. Intended Use and Limits**

*   **Intended Use:** This playbook is designed as a **decision-support tool** for the marketing and SEO teams. It acts as an early warning system to highlight decaying content and prioritize editorial pipelines efficiently.
*   **Limits:** The model relies solely on Google Search Console data. It does **not** understand content quality, backlink profiles, or actual on-page conversion rates (e.g., sign-ups or purchases). Therefore, a page flagged as "underperforming" in CTR might still be highly valuable for other business metrics.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**3. Human Review and The No-Go List**

*   **Human-in-the-Loop:** All recommended actions (Refresh, Audit, Prune) must be reviewed by an SEO Manager before execution. The model suggests the queue; humans make the final call.
*   **The No-Go List (What NOT to automate):**
    *   **Auto-Deletion:** Never automatically delete or unpublish pages based on this model's score.
    *   **Core Brand Pages:** Homepage, pricing pages, and legal policies are strictly excluded from automated pruning pipelines, regardless of their CTR.
    *   **Auto-Redirects:** 301 redirects must not be mapped automatically to prevent redirect loops and loss of contextual relevance.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**4. Monitoring and Retrain Triggers**

*   **Performance Monitoring:** The Data Science team will sample 100 predictions monthly and compare them against human SEO expert reviews to track the model's precision.
*   **Retrain Triggers:**
    1. **Data Drift:** Retrain the model if a major Google Core Algorithm Update is announced, as user search behavior and SERP layouts often change fundamentally.
    2. **Metric Drop:** Retrain if the directional F1-Score on the monthly human-reviewed sample drops below 0.85.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import userdata

# 1. Ensure export directories exist
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 2. Re-create the verified queue logic (Simulation of the model's output for the playbook)
# Fetching the same dataset used in Week 5 & 6
hf_token = userdata.get('HF_TOKEN')
fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df_fact = pd.read_parquet(fact_path, storage_options={"token": hf_token})

df = df_fact.groupby(['client_hash_id', 'content_hash_id']).agg({
    'gsc_impressions': 'sum',
    'gsc_clicks': 'sum',
    'gsc_avg_position': 'mean'
}).reset_index()

df['ctr'] = df['gsc_clicks'] / df['gsc_impressions'].replace(0, 1)

# Apply Playbook Logic
def assign_action(row):
    if row['gsc_impressions'] > 500 and row['gsc_avg_position'] <= 15 and row['ctr'] < 0.01:
        return 'R-01: Content Refresh'
    elif row['gsc_impressions'] > 100 and row['gsc_avg_position'] <= 3 and row['ctr'] < 0.05:
        return 'R-02: Technical SEO Audit'
    elif row['gsc_impressions'] < 50 and row['gsc_clicks'] == 0:
        return 'R-03: Prune / Consolidate'
    return 'No Action'

df['playbook_action'] = df.apply(assign_action, axis=1)

# 3. Filter for actionable items and export
action_queue = df[df['playbook_action'] != 'No Action'].copy()
action_queue = action_queue.sort_values(by='gsc_impressions', ascending=False)

export_path = 'work/outputs/action_queue.csv'
action_queue.to_csv(export_path, index=False)

print(f"--- PLAYBOOK EXPORT COMPLETE ---")
print(f"Total actionable pages flagged: {len(action_queue)}")
print(f"Exported successfully to: {export_path}")
print("Note: As per CI rules, this CSV data file will not be committed to Git.")

--- PLAYBOOK EXPORT COMPLETE ---
Total actionable pages flagged: 258412
Exported successfully to: work/outputs/action_queue.csv
Note: As per CI rules, this CSV data file will not be committed to Git.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.